In [13]:
import os 
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

current_dir = "/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp"
folders = [os.path.join(current_dir, d) for d in os.listdir(current_dir) if os.path.isdir(os.path.join(current_dir, d))]

def load_data_from_folders(folders):
    X_data = []
    Y_dist_data = []
    Y_norm_data = []
    
    for folder in folders:
        print(folder)
        X_path = os.path.join(folder, 'X_processed.npy')
        Y_dist_path = os.path.join(folder, 'Y_dist.npy')
        Y_norm_path = os.path.join(folder, 'Y_norm.npy')
        if os.path.exists(X_path) and os.path.exists(Y_dist_path) and os.path.exists(Y_norm_path):
            X_data.append(np.load(X_path))
            Y_dist_data.append(np.load(Y_dist_path))
            Y_norm_data.append(np.load(Y_norm_path))
    return np.concatenate(X_data), np.concatenate(Y_dist_data), np.concatenate(Y_norm_data)

X, Y_dist, Y_norm = load_data_from_folders(folders)
print("Raw shapes:", X.shape, Y_dist.shape, Y_norm.shape)

# X: (N, 11, 2, 25) -> (N, 11, 2) by taking median over the 25 values
# dim 2, index 0 = attenuation, index 1 = phase
X = np.median(X, axis=-1)  # (N, 11, 2)
print("Processed X shape:", X.shape)

# Convert to float32 tensors
X_tensor = torch.from_numpy(X).float()           # (N, 11, 2)
Y_dist_tensor = torch.from_numpy(Y_dist).float() # (N, 4)
Y_norm_tensor = torch.from_numpy(Y_norm).float() # (N, 4, 3)

dataset = TensorDataset(X_tensor, Y_dist_tensor, Y_norm_tensor)

# 80/20 train/val split
val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Train samples: {train_size}, Val samples: {val_size}")

# Sanity check a batch
X_batch, Y_dist_batch, Y_norm_batch = next(iter(train_loader))
print("Batch shapes:", X_batch.shape, Y_dist_batch.shape, Y_norm_batch.shape)


/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east2_1
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east2_0
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east1_6
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/northeast0
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east1_1
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/northeast1
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east1_0
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/northeast6
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east2_2
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east2_3
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/northeast3
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east1_2
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/northeast4
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east1_5
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/northeast5
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/east1_4
/Users/manavjeet/git/RFImaging/Rovering/OCS/T_exp/nort

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim=22, hidden_dim=64, output_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        # x: (B, 11, 2) -> flatten to (B, 22)
        return self.net(x.flatten(start_dim=1))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

NUM_EPOCHS = 50

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for X_b, Y_dist_b, _ in train_loader:
        X_b, Y_dist_b = X_b.to(device), Y_dist_b.to(device)
        pred = model(X_b)
        loss = loss_fn(pred, Y_dist_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(X_b)
    train_loss /= train_size

    # --- Val ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, Y_dist_b, _ in val_loader:
            X_b, Y_dist_b = X_b.to(device), Y_dist_b.to(device)
            pred = model(X_b)
            val_loss += loss_fn(pred, Y_dist_b).item() * len(X_b)
    val_loss /= val_size

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | Val RMSE: {val_loss**0.5:.4f} m")
